# Session 01: Python Runtime and Professional Setup

**Notebook 1 of 2: Concepts** · used during the live session and for revision afterwards (about 45 minutes on your own)

By the end of this notebook you will be able to:

1. **Explain** what happens between typing `python main.py` and seeing output, and which errors Python can and cannot catch before your code runs.
2. **Explain** why Python projects need virtual environments, and inspect which environment your code is running in.
3. **Describe** what `pyproject.toml`, `uv.lock`, `.python-version` and `.venv/` each do, and which of them belong in git.
4. **Use** the core `uv` commands to create a project, manage dependencies and run code reproducibly.
5. **Debug** the most common Python setup failure: a package that is installed, but not where the running interpreter is looking.
6. **Configure** access to an LLM provider (OpenAI by default, or Anthropic, OpenRouter or Ollama) through environment variables, without putting secrets in code.
7. **Generate** text embeddings with OpenAI (the default) or locally with fastembed, and **rank** documents by cosine similarity.

> **Before you start:** this notebook must run on the course environment's kernel. From the course repository root, run `uv sync` once. Then either run `uv run jupyter lab`, or open the notebook in VS Code and choose **Select Kernel → Python Environments → `.venv`** (the one inside the course repository). Part 3 shows you how to confirm you got it right.

## The engineering problem

You've just joined the platform team at **Orbital Logistics** (a fictional company we'll use throughout the course). The team keeps a folder of Python operations scripts. The onboarding note says *"install Python, then `pip install` whatever's missing."*

By lunch:

- One script fails with `ModuleNotFoundError`, even though you installed the package ten minutes ago.
- Another script works on a colleague's laptop but crashes on yours, because they have an older version of a library installed and you have a newer one.
- Upgrading that library for script B breaks script A, which needs the old version.

None of these is a Python *language* problem. They are **runtime and environment** problems, and they are where experienced developers lose the most time when they start with Python.

What surprises people coming from other ecosystems is this: **in npm, `node_modules` is per project by default. In Python, the default is the opposite.** A bare `pip install` puts packages into whichever interpreter `pip` belongs to, shared by everything that uses that interpreter. Isolation is something you opt into. That one fact explains most Python setup pain, and this session is about opting in properly.

## Part 1: What happens when Python runs

You've probably heard "Python is interpreted." That's half the story, and the missing half explains behaviour you'll run into in your first week.

When you run a file, CPython (the standard Python implementation) **first compiles the whole file to bytecode**: compact instructions for a virtual machine. It does this automatically, in memory, every time. Then the interpreter **executes that bytecode instruction by instruction**. It's the same broad idea as the JVM or the CLR, except there is no separate build step. Nothing like `javac` or `tsc` examines your whole program before it starts.

![What happens when Python runs your code](../../../assets/diagrams/python-execution-model.svg)

**How this works**

- When you run `main.py`, the whole file is compiled first. A **syntax** error anywhere stops everything before a single line runs.
- The interpreter then executes the bytecode. Errors in **names, types and attributes** only appear when the line containing them actually runs. A misspelled variable in a branch nobody exercised will sit there silently.
- Modules you `import` are compiled once and cached as `.pyc` files in `__pycache__/`, so later runs skip recompiling them. This is purely an optimisation: deleting `__pycache__` is always safe.
- `import rich` makes Python search the directories on `sys.path` **in order** and stop at the first match. The first import runs the module's top-level code; later imports reuse the module object already cached in `sys.modules`.
- The third location, `site-packages`, belongs to **whichever interpreter is running**. Run the same code with a different interpreter and a different set of packages is visible. That is the root of most "but I installed it!" bugs, which Part 6 takes apart.

### Which Python am I running?

Before anything else, a professional habit: when something behaves strangely, check *which* interpreter is running. `sys` is the standard-library module that exposes the running interpreter.

**Predict:** this notebook's kernel is supposed to be the course environment. What do you expect the `Interpreter` line to contain?

In [ ]:
import sys

print("Implementation:", sys.implementation.name)
print("Version:       ", sys.version.split()[0])
print("Interpreter:   ", sys.executable)

**Explain:** `cpython` is the reference implementation, the one you download from python.org and the one uv installs. Others exist (PyPy, for example), but CPython is what this course and most production systems use. The `Interpreter` path should point inside a `.venv` folder in the course repository. If it points somewhere like `C:\Python313\python.exe` or `/usr/bin/python3`, the notebook is running on the wrong kernel. Fix that before you continue.

### Seeing the bytecode, and what isn't in it

You will rarely need to read bytecode. Looking at it once, though, makes the execution model concrete. The standard-library `dis` module disassembles a function:

In [ ]:
import dis


def shipping_cost(weight_kg: float, rate_per_kg: float) -> float:
    return weight_kg * rate_per_kg


dis.dis(shipping_cost)

You'll see instructions such as `LOAD_FAST_LOAD_FAST` (load two local variables; older versions show two separate `LOAD_FAST` instructions), `BINARY_OP` (the `*`) and `RETURN_VALUE`. Instruction names change between Python versions, which is one reason `.pyc` files are version-specific.

Notice what is **missing**: nothing in the bytecode checks that the arguments are floats. The annotations `: float` and `-> float` are **type hints**. They are stored as metadata for editors and tools, and the interpreter does not enforce them.

**Predict:** what does `shipping_cost("3", 2)` return? In Java or C# this would not compile. Here?

In [ ]:
shipping_cost("3", 2)

**Explain:** `"3" * 2` is valid Python. It repeats the string, giving `'33'`. The hints did not stop the wrong type, and no error was raised: you got a silently wrong answer. This is not a reason to avoid type hints. It is a reason to understand them correctly. Hints are for **tools** (your editor, linters, type checkers). Data arriving from outside your program needs **runtime validation**, which is what Pydantic (Session 06) is for.

**Engineering Insight:** in a typed language, the compiler is your first test. In Python, linters (Ruff, Session 04), type checkers and tests have to fill that role. Plan for that from day one.

### Compile-time versus run-time errors

**Predict:** the function below refers to `SLA_DAYS`, which is never defined. Will defining the function raise an error?

In [ ]:
def overdue_tickets(tickets):
    return [ticket for ticket in tickets if ticket["age_days"] > SLA_DAYS]


print("Function defined without complaint.")

Defining it is fine, because Python only looks up `SLA_DAYS` when that line runs. Now **call** it. The call is *expected* to fail. We catch the exception and print the real traceback, so that "Run All" keeps going. In a script, this traceback is what you would see just before the program exits:

In [ ]:
import traceback

try:
    overdue_tickets([{"ticket_id": "INC-1042", "age_days": 9}])
except NameError:
    traceback.print_exc()

Compare that with a **syntax** error. To keep this notebook running, we compile a broken snippet with the built-in `compile()` and catch the error, instead of putting broken syntax in a cell:

In [ ]:
broken_source = "def overdue_tickets(tickets:\n    return []\n"

try:
    compile(broken_source, "broken_module.py", "exec")
except SyntaxError as error:
    print(f"{type(error).__name__} in {error.filename}, line {error.lineno}: {error.msg}")

**Explain:**

- A `SyntaxError` is caught at compile time, before *any* of the module runs, including lines above the mistake.
- A `NameError` (or `TypeError`, `AttributeError`, `KeyError` …) is only raised when the bad line executes. "The script ran fine once" tells you nothing about paths that didn't execute.

**Common Mistake:** treating a successful run as proof that a code path works. Tests that exercise each branch are what give you the confidence a compiler would.

### The bytecode cache

When a module is imported, CPython writes its compiled bytecode to `__pycache__/`. The file name includes an interpreter tag:

In [ ]:
import importlib.util

print(importlib.util.cache_from_source("ticket_rules.py"))

The `cpython-313` tag means "compiled by CPython 3.13". A different interpreter version simply ignores that file and compiles its own. That's why `__pycache__/` and `*.pyc` are always in `.gitignore`: they are machine-local build output, not source.

Now watch it happen. The next cell writes a small module to a temporary folder, imports it **twice**, and lists what Python cached. **Predict:** how many times will the module's `print` run?

In [ ]:
import importlib
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as workdir:
    module_file = Path(workdir) / "ticket_rules.py"
    module_file.write_text("SLA_DAYS = 5\nprint('ticket_rules: top-level code is running')\n")

    sys.path.insert(0, workdir)
    importlib.invalidate_caches()
    try:
        import ticket_rules

        import ticket_rules  # noqa: F811  (second import on purpose)

        print("SLA_DAYS =", ticket_rules.SLA_DAYS)
        print("Cached files:", [path.name for path in (Path(workdir) / "__pycache__").iterdir()])
    finally:
        sys.path.remove(workdir)
        sys.modules.pop("ticket_rules", None)

**Explain:** the top-level code ran **once**. The second `import` found `ticket_rules` already in `sys.modules` and reused it. This matters later: module-level code (connecting to a database, reading configuration) runs at import time, exactly once per process. (The `finally` block cleans up so that this demo leaves no hidden state in the notebook.)

### How this compares with what you know

| | Java | C# | JavaScript / TypeScript (Node) | Python (CPython) |
|---|---|---|---|---|
| Build step | `javac` → `.class` | `dotnet build` → IL `.dll` | `tsc` (TS only) | None: compiled automatically at run/import time |
| Runtime | JVM (JIT) | CLR (JIT) | V8 (JIT) | Bytecode interpreter (an experimental JIT exists in 3.13 but is off by default) |
| Caught before running | Syntax, names, types | Syntax, names, types | Syntax; types with TS | **Syntax only** |
| Cached artefact | `.class` / `.jar` | `.dll` | n/a | `__pycache__/*.pyc` |

Don't read "no compiler" as "Python is a toy". It's a trade-off: faster feedback and less ceremony, in exchange for responsibilities that move to linters and tests.

## Part 2: Python versions

Python ships a new feature release every October, and each one is supported with fixes for about five years. Versions matter for three practical reasons:

- **Language and standard library:** new syntax and modules appear, and deprecated ones are removed.
- **Compiled packages:** many libraries (NumPy, Pydantic's core, database drivers) ship pre-built binaries ("wheels") *per Python version*. A brand-new Python release sometimes has to wait for its wheels.
- **Reproducibility:** "works on 3.13" is a fact. "Works on Python" is not.

This course uses **Python 3.13**, chosen because every library in the course installs and runs on it. Each project records that choice in two places, which you'll see in Part 4: `requires-python` in `pyproject.toml`, and the `.python-version` file.

`sys.version_info` is a tuple, and tuples compare element by element, which makes version checks readable:

In [ ]:
print(sys.version_info)
print("Meets the course minimum (3.13):", sys.version_info >= (3, 13))

**Modify:** change `(3, 13)` to `(3, 14)` and to `(3, 12, 5)`, then re-run. **Explain:** why is comparing tuples safer than comparing strings such as `"3.13" >= "3.9"`? (Try that string comparison too.)

## Part 3: Virtual environments

Back to the lunchtime problem: two scripts need different versions of the same library. A **virtual environment** gives each project its own private `site-packages` directory, while reusing one installed interpreter.

![Global versus project environments](../../../assets/diagrams/virtual-environment-isolation.svg)

**How this works**

- On the left, both projects install into the same global `site-packages`, and Python can hold only one copy of `pydantic` there. Whichever install ran last wins, and `billing-service`, written for the v1 API, fails at import time.
- On the right, each project gets its own `.venv/`, created by `uv sync` from that project's `pyproject.toml` and `uv.lock`. The two projects can now depend on incompatible versions of the same library.
- The interpreter itself is shared. A virtual environment is a separate *package* directory plus a pointer back to a base interpreter, not a full copy of Python. That's why creating one takes milliseconds, and why it is safe to delete and recreate.
- `uv.lock` is what makes the right-hand side reproducible: a teammate or CI machine running `uv sync` gets exactly the same versions, much like `package-lock.json`.
- Failure case: running `python app.py` in a terminal where the global interpreter comes first on `PATH` silently puts you back in the left-hand world. `uv run` avoids this by always using the project's environment.

### Technical details

A virtual environment is just a directory containing:

- `pyvenv.cfg`, a small text file that records which base interpreter it points to;
- `Scripts/` (Windows) or `bin/` (macOS/Linux), holding a `python` executable plus any command-line tools you install (`ruff`, `pytest` …);
- `Lib/site-packages/` (Windows) or `lib/python3.13/site-packages/` (macOS/Linux), holding the installed packages.

"Activating" an environment (`source .venv/bin/activate`, or `.venv\Scripts\Activate.ps1` on Windows) only puts its `Scripts/`/`bin/` folder first on your `PATH`, nothing more. `uv run` makes activation unnecessary, which is why this course rarely uses it.

Inside a running interpreter, `sys.prefix` is the environment you're in, and `sys.base_prefix` is the interpreter it was created from. If they differ, you're in a virtual environment.

**Predict:** will `In a virtual environment` print `True` or `False` in this notebook?

In [ ]:
import sysconfig

print("sys.prefix:              ", sys.prefix)
print("sys.base_prefix:         ", sys.base_prefix)
print("In a virtual environment:", sys.prefix != sys.base_prefix)
print("Packages install into:   ", sysconfig.get_paths()["purelib"])

It should be `True`, with `sys.prefix` ending in `.venv`. Now look inside `pyvenv.cfg`:

In [ ]:
venv_config = Path(sys.prefix) / "pyvenv.cfg"

if venv_config.exists():
    print(venv_config.read_text())
else:
    print("No pyvenv.cfg: this kernel is not running in a virtual environment.")

**Explain:** `home` points at the base interpreter's folder; this is the "pointer back" from the diagram. Because uv created this environment, you'll also see a `uv` line recording its version.

Finally, the packages *this* environment can see. **Predict** which of these are installed. Remember that the course environment only declares what the notebooks need.

In [ ]:
from importlib.metadata import PackageNotFoundError, version

for distribution in ["ipykernel", "jupyterlab", "django"]:
    try:
        print(f"{distribution:<12} {version(distribution)}")
    except PackageNotFoundError:
        print(f"{distribution:<12} not installed in this environment")

`django` isn't installed, and that's correct: nothing in this course's `pyproject.toml` declares it. Each environment contains only what its project declares. Your own projects work the same way: in Session 07 you'll add FastAPI to *your* project with `uv add`, and it will be installed in that project's own environment.

| | npm | Maven / Gradle | .NET | Python with uv |
|---|---|---|---|---|
| Where dependencies live | `node_modules/` per project | Shared cache (`~/.m2`, Gradle cache) plus a per-build classpath | Shared NuGet cache, resolved per project | `.venv/` per project (uv also keeps a shared download cache) |
| Default without a tool | Per project | Per project | Per project | **Global**, which is why you opt in |
| Disposable? | Yes (`rm -rf node_modules`) | n/a | `bin/`, `obj/` | Yes (delete `.venv/`, run `uv sync`) |

## Part 4: Managing projects with uv

Python's packaging history left a lot of overlapping tools: `pip` installs, `venv` and `virtualenv` create environments, `pip-tools` locks, `pyenv` manages interpreters, and `pipenv` and `poetry` combine some of these. You will meet all of them in other people's repositories (the course `docs/` folder has guides for `venv` and Conda).

**uv** combines the whole workflow in one fast tool: it installs Python versions, creates environments, resolves and locks dependencies, and runs your code. Think of it as `nvm` + `npm` + `npx` in one, or Maven plus a JDK manager.

![Anatomy of a uv project](../../../assets/diagrams/uv-project-structure.svg)

**How this works**

- `pyproject.toml` is the standard Python project file (defined by PEP 621). It says *what you want*: a name, a Python requirement, and dependency **ranges** such as `rich>=15.0.0`. You normally edit it through `uv add` and `uv remove`.
- `uv lock` resolves those ranges into one consistent set of exact versions, including transitive dependencies you never named, plus a hash for each file, and writes `uv.lock`. Like `package-lock.json`, it is generated (never edit it by hand) **and committed**.
- `uv sync` makes `.venv/` match `uv.lock` exactly: it installs what's missing and removes what isn't listed. That's why `.venv/` is disposable and git-ignored; any machine can rebuild it.
- `.python-version` tells uv which interpreter to base the environment on. If that version isn't installed, uv downloads it.
- Day to day you rarely call `uv lock` or `uv sync` yourself: `uv add` performs all three steps, and `uv run` checks the lock and syncs before running. CI servers call `uv sync --locked` explicitly (see Part 7).

### The core commands

| Command | What it does | Closest equivalent you know |
|---|---|---|
| `uv python install 3.13` | Download and install a Python interpreter | `nvm install`, SDKMAN |
| `uv init --no-package --python 3.13 NAME` | Create a new project (`pyproject.toml`, `.python-version`, `main.py`, `README.md`, plus git setup) | `npm init`, `dotnet new console` |
| `uv add rich` | Add a runtime dependency (updates pyproject, lock and `.venv`) | `npm install rich` |
| `uv add --dev ruff` | Add a development-only dependency | `npm install --save-dev` |
| `uv remove rich` | Remove a dependency | `npm uninstall` |
| `uv lock` | Re-resolve and rewrite `uv.lock` | `npm install --package-lock-only` |
| `uv sync` | Make `.venv` match the lock exactly | `npm ci` |
| `uv run main.py` | Run code in the project environment, syncing first if needed | `npm run`, `npx` |
| `uv tree` | Show the dependency tree | `npm ls`, `mvn dependency:tree` |

> **Why `--no-package`?** In uv 0.13, a plain `uv init` creates a *packaged* project with a `src/` folder and a build backend. That layout is useful, and you will build one in Session 04. For now we want the simplest possible project, a single `main.py`. Writing `--no-package --python 3.13` explicitly also gives you the same result whichever uv version or default Python is on your machine.

Mapping the files to ecosystems you know:

| Purpose | npm | Maven | .NET | uv |
|---|---|---|---|---|
| Declared dependencies | `package.json` | `pom.xml` | `.csproj` | `pyproject.toml` |
| Exact resolved versions | `package-lock.json` | (none by default) | `packages.lock.json` | `uv.lock` |
| Installed packages | `node_modules/` | local repository + classpath | NuGet cache | `.venv/` |
| Runtime version | `.nvmrc` | toolchains | `global.json` | `.python-version` |

### The lockfile is the truth

The course repository itself is a uv project, and this notebook's kernel was built from it. Let's read its `pyproject.toml` and `uv.lock`. Both are TOML, which the standard-library `tomllib` module can parse.

**Predict:** the course `pyproject.toml` declares **seven** dependencies: three for Jupyter (including `ipywidgets`, which draws progress bars), and four you'll meet in Parts 8 and 9. How many packages do you expect in the lockfile?

In [ ]:
import tomllib

repo_root = Path.cwd().parents[2]  # notebooks -> session-01 -> sessions -> repository root

if not (repo_root / "pyproject.toml").exists():
    print(f"Expected the course pyproject.toml in {repo_root}.")
    print("Open this notebook from inside the course repository so relative paths resolve.")
else:
    with (repo_root / "pyproject.toml").open("rb") as file:
        project = tomllib.load(file)["project"]
    print("requires-python:", project["requires-python"])
    print("declared:       ", project["dependencies"])

    with (repo_root / "uv.lock").open("rb") as file:
        lock = tomllib.load(file)
    locked_versions = {package["name"]: package["version"] for package in lock["package"]}
    print("locked packages:", len(locked_versions))

Seven declared dependencies pulled in more than a hundred packages. Those are **transitive** dependencies, and every one of them is pinned in the lock. Now check that the environment really matches the lock. `tornado` is a dependency of `ipykernel`, one you never asked for directly:

In [ ]:
for distribution in ["ipykernel", "jupyterlab", "tornado"]:
    locked = locked_versions[distribution]
    installed = version(distribution)
    status = "match" if locked == installed else "MISMATCH, run uv sync"
    print(f"{distribution:<12} locked {locked:<10} installed {installed:<10} {status}")

**Explain:** `pyproject.toml` said `ipykernel>=7.4.0`, a range. The lock chose one exact version, and `uv sync` installed exactly that. A colleague who clones the repository next month gets the same versions, even if newer ones have been released since. Upgrades happen when you decide (`uv lock --upgrade-package NAME`), not by accident.

**Version specifiers you'll see in `pyproject.toml`:**

| Specifier | Meaning |
|---|---|
| `rich>=15.0.0` | 15.0.0 or newer (what `uv add` writes by default) |
| `rich>=15,<16` | Any 15.x release |
| `rich~=15.0` | "Compatible release": `>=15.0, <16` |
| `rich==15.0.0` | Exactly this version (rarely needed in `pyproject.toml`; the lock already pins) |

**Engineering Insight:** for an *application*, keep ranges sensible and let the lockfile pin. For a *library* that other people install, keep ranges broad, because over-tight pins in libraries cause conflicts for everyone downstream.

### Try it in your terminal

This part happens outside the notebook. Open a terminal in a scratch folder (not inside the course repository) and work through these steps. **Before each command, predict what it will create or change.**

```bash
uv init --no-package --python 3.13 scratch-demo
cd scratch-demo
uv run main.py          # Predict: which new files and folders appear? (list them afterwards)
uv add rich             # Then open pyproject.toml. What changed? Does uv.lock list only rich?
uv tree                 # Where do markdown-it-py, mdurl and pygments come from?
uv add --dev ruff       # Look for the new [dependency-groups] section
uv remove rich          # Check pyproject.toml, uv.lock and .venv again
```

**Modify:** delete the `.venv` folder, then run `uv run main.py` again. **Explain:** why does this work, and why is it fast?

You'll do this properly, in a project you keep, in the workbook assignment.

## Part 5: Running code: scripts, modules and notebooks

There are three ways you'll run Python code in this course:

| You want to … | Command | What it means |
|---|---|---|
| Run a file | `uv run main.py` | Execute `main.py` as the main program |
| Run a module by import name | `uv run python -m json.tool orders.json` | Find `json.tool` on `sys.path` and run it as the main program |
| Run a tool installed in the environment | `uv run ruff check .` | Run the `ruff` command from `.venv/Scripts` (or `.venv/bin`) |

The `-m` form matters more than it looks. `python -m pip` or `python -m pytest` guarantees the tool belongs to *that* interpreter, which avoids the "wrong pip" trap from the introduction.

### `if __name__ == "__main__":`

Every module has a `__name__`. When a file is run as the main program, its `__name__` is `"__main__"`. When it's imported, `__name__` is the module's name. This idiom lets one file be both an importable module and a runnable script, roughly what `public static void main` gives you, except that it's opt-in.

**Predict** the two lines this cell prints:

In [ ]:
import subprocess

with tempfile.TemporaryDirectory() as workdir:
    Path(workdir, "report.py").write_text('print(f"report.py sees __name__ = {__name__!r}")\n')

    as_script = subprocess.run(
        [sys.executable, "report.py"], cwd=workdir, capture_output=True, text=True, check=True
    )
    as_import = subprocess.run(
        [sys.executable, "-c", "import report"], cwd=workdir, capture_output=True, text=True, check=True
    )

print("Run as a script:", as_script.stdout.strip())
print("Imported:       ", as_import.stdout.strip())

**Explain:** the same file printed different values depending on *how* it was loaded. That's why `main.py` from `uv init` ends with `if __name__ == "__main__": main()`. Importing the file, for example from a test in Session 04, must not run the program.

What about this notebook? Run `print(__name__)` in a new cell. A notebook runs as the main program, so you'll see `__main__`.

### Notebooks and kernels

A Jupyter notebook is a document connected to a **kernel**, which is a running Python process. The kernel is an interpreter, so everything in Part 3 applies: the kernel must be the project's `.venv` interpreter, or imports will fail.

- **VS Code:** use **Select Kernel** (top right of the notebook). For `.py` files, use **Python: Select Interpreter** in the Command Palette and choose the `.venv` inside your project. VS Code shows the selected interpreter in the status bar.
- **JupyterLab:** start it with `uv run jupyter lab` from the project folder, and the kernel uses that project's environment.
- **Hidden state:** a kernel remembers every cell you have run, in the order you ran them. Before trusting a notebook, use **Restart Kernel and Run All**. If it only works when cells run out of order, it doesn't work.

## Part 6: Debugging lab: "But I installed it!"

This is the setup failure you are most likely to meet, both in this course and in your first Python job. We'll debug it the way we'll debug everything in this course: **Observe → Hypothesize → Inspect → Fix → Verify**.

### Observe

A teammate says: *"I ran `uv add orbital-metrics` and it succeeded, but my script fails."* We'll reproduce the failure, again printing the real traceback instead of letting it stop the notebook:

In [ ]:
try:
    import orbital_metrics  # noqa: F401
except ModuleNotFoundError:
    traceback.print_exc()

**Reading a traceback.** Python tracebacks are read **bottom up**:

- The **last line** is the exception type and message: `ModuleNotFoundError: No module named 'orbital_metrics'`. Start there.
- The lines above it show *where* the error happened, with the most recent call last. Here the frame points at the `import` line in this cell. In a real script you'd see the file name and line number of the failing import.
- For import errors, the important fact is missing from the traceback: **which interpreter was searching**. You have to inspect that yourself.

### Hypothesize

There are three usual causes:

1. **Not installed at all**, for example because `uv add` ran in a different project folder.
2. **Installed into a different environment** than the one running the code. This is the most common cause by far: running `python script.py` with a global interpreter, or a VS Code or notebook kernel pointing at the wrong environment.
3. **The install name is not the import name.** The name you install (the *distribution*) can differ from the name you import. Common examples:

   | You install | You import |
   |---|---|
   | `python-dotenv` | `dotenv` |
   | `beautifulsoup4` | `bs4` |
   | `PyYAML` | `yaml` |
   | `scikit-learn` | `sklearn` |

### Inspect

A small helper that answers the questions the traceback doesn't. `importlib.util.find_spec` asks "could this interpreter import this module, and from where?" without actually importing it:

In [ ]:
def diagnose_import(module_name: str) -> None:
    spec = importlib.util.find_spec(module_name)
    print(f"Interpreter : {sys.executable}")
    print(f"In a venv   : {sys.prefix != sys.base_prefix}")
    if spec is None:
        print(f"Module      : {module_name!r} is NOT importable by this interpreter")
    else:
        print(f"Module      : {module_name!r} found at {spec.origin}")
    print()


diagnose_import("json")  # standard library
diagnose_import("ipykernel")  # third-party, installed in .venv
diagnose_import("orbital_metrics")  # the failing one

Read the three results. `json` resolves to the base interpreter's standard library, `ipykernel` resolves to the `.venv` `site-packages`, and `orbital_metrics` resolves nowhere. (It doesn't exist; we invented it for this lab.) For a real package, the `Interpreter` line usually tells you immediately whether cause 2 is the culprit.

### Fix

| Symptom | Likely cause | Fix |
|---|---|---|
| `uv add X` succeeded, but `python main.py` raises `ModuleNotFoundError` | The global interpreter is running, not `.venv` | Run `uv run main.py` |
| VS Code underlines the import, but `uv run` works | VS Code has the wrong interpreter selected | **Python: Select Interpreter** → the project's `.venv` |
| Notebook import fails, but scripts work | The notebook kernel is not the project's `.venv` | **Select Kernel** → the project's `.venv`, then restart |
| `pip install X` "worked", but nothing changed | That `pip` belongs to another interpreter | Use `uv add X` (or at least `python -m pip`) |
| Installed `python-dotenv`, but `import python_dotenv` fails | Install name ≠ import name | Check the package's documentation for its import name |
| Windows: typing `python` opens the Microsoft Store | Windows "App execution aliases" | Use `uv run …`, or turn off the aliases in Settings |
| Windows: `Activate.ps1` "cannot be loaded because running scripts is disabled" | PowerShell execution policy | Use `uv run` (no activation needed), or `Set-ExecutionPolicy -Scope CurrentUser RemoteSigned` |

### Verify

After a fix, re-run the *original* failing command, not a different one, and confirm with `diagnose_import` (or `uv run python -c "import sys; print(sys.executable)"`) that the expected interpreter is the one running.

**Challenge:** in your own terminal, run `python -c "import sys; print(sys.executable)"` and then `uv run python -c "import sys; print(sys.executable)"` inside the scratch project. Explain the difference in one sentence.

## Part 7: Production considerations

The habits from this session carry straight into professional work:

- **Commit** `pyproject.toml`, `uv.lock`, `.python-version` and your source. **Never commit** `.venv/`, `__pycache__/` or `.env`. The `.gitignore` that `uv init` creates already covers the first two.
- **CI and deployment install from the lock, strictly.** `uv sync --locked` fails if `uv.lock` doesn't match `pyproject.toml`, for example because someone edited dependencies by hand and forgot to relock. The failure looks like this:

  ```text
  error: The lockfile at `uv.lock` needs to be updated, but `--locked` was provided.

  hint: To update the lockfile, run `uv lock`.
  ```

  That error is a feature: it stops an unreviewed dependency change from reaching production.
- **Separate runtime and development dependencies.** Linters, test runners and notebook tools go in the dev group (`uv add --dev`). A production image can then install with `uv sync --locked --no-dev`.
- **Upgrade deliberately.** `uv lock --upgrade-package rich` upgrades one package within its allowed range. Review the `uv.lock` diff the way you'd review a code diff.
- **Supply-chain hygiene.** Hashes in `uv.lock` protect you from a *tampered* file, but not from installing the *wrong* package. Typosquatted names (`reqeusts` instead of `requests`) exist on PyPI. Check a package's name and project page before you `uv add` it, and be especially careful when an AI assistant suggests a package you've never heard of (Session 13).

## Part 8: Preparing your LLM provider (preview)

You won't call an LLM in earnest until Session 09. Provider access is still part of *setup*, though, and it follows the same principle as the rest of this session: **configuration belongs to the environment, not the code.** Dependencies live in `.venv`; credentials and model choices live in environment variables. Setting this up now means Session 09 starts with code, not sign-up forms. (Self-study, about 15 minutes. It is only previewed in the live session.)

The course uses **OpenAI** by default, with the model **`gpt-5.6-terra`**. You can use Anthropic, OpenRouter or a local Ollama model instead, and everything in this part works with any of them.

![One application, several LLM providers](../../../assets/diagrams/llm-provider-options.svg)

**How this works**

- Your code doesn't hard-code a provider. It reads `LLM_PROVIDER`, `LLM_MODEL` and the matching API key from the environment, so switching provider is a configuration change rather than a code change.
- The `openai` SDK speaks OpenAI's API format, which OpenRouter and Ollama also implement (they are *"OpenAI-compatible"*). To reach them you change only the client's `base_url` and the model ID. The calls stay the same.
- **OpenRouter** is a gateway: one account and one key give you models from many vendors, which it names `vendor/model`, for example `openai/gpt-5.6-terra` or `anthropic/claude-haiku-5.5`.
- **Ollama** runs open-weight models on your own machine: free, private and usable offline. The trade-off is that a laptop-sized model such as `llama3.2` is noticeably less capable than hosted models, and it needs a few GB of RAM and disk.
- **Anthropic** has its own SDK, `anthropic`, with a slightly different request and response shape. Claude models are also reachable through OpenRouter, under a *different* ID. That is the caveat at the bottom of the diagram.

### Choosing a provider

| Provider | Use it when | Watch out for |
|---|---|---|
| **OpenAI** (default) | You want the path the course examples are written for | Paid per token; set a monthly budget in the dashboard |
| **Anthropic** | Your team uses Claude, or you want to compare behaviour | Different SDK and response shape (shown below) |
| **OpenRouter** | You want to try many models with one key and one bill | An extra party sees your requests; model IDs carry a vendor prefix |
| **Ollama** | No budget, no internet, or data that must not leave your machine | Smaller models give weaker answers, and speed depends on your hardware |

### Model tiers and what they cost

Each provider sells several *tiers* of model. Anthropic's are a good illustration. This course's Claude example is **Claude Haiku 5.5** (`claude-haiku-5-5`), Anthropic's fast, low-cost model, which is plenty for the extraction and classification work in this course. **Claude Opus 5.5** (`claude-opus-5-5`) is Anthropic's most capable model, intended for hard reasoning, long agentic tasks and complex coding.

| Model | ID at Anthropic | ID at OpenRouter | Input (USD per 1M tokens) | Output (USD per 1M tokens) |
|---|---|---|---:|---:|
| Claude Haiku 5.5 | `claude-haiku-5-5` | `anthropic/claude-haiku-5.5` | 0.10 | 0.50 |
| Claude Opus 5.5 | `claude-opus-5-5` | `anthropic/claude-opus-5.5` | 4.00 | 20.00 |

List prices in October 2026. Haiku's prices apply to prompts up to 100K tokens; check the [pricing page](https://platform.claude.com/docs/en/about-claude/pricing) for current rates.

**The cost factor is 40×.** A workload that costs USD 5 a month on Haiku costs about USD 200 on Opus. To make that concrete: a request with a 2,000-token prompt and a 500-token answer costs about USD 0.00045 on Haiku and USD 0.018 on Opus, so 10,000 such requests cost roughly USD 4.50 against USD 180. (Both models can "think" before answering, and thinking tokens are billed as output, so real requests often cost somewhat more than this estimate.)

**Engineering Insight:** start with the cheapest model that does the job, and measure before moving up a tier. Reach for Opus when the cheaper model demonstrably fails on *your* task, not because a bigger model feels safer. Because the model ID is configuration, trying Opus later is a one-line change in `.env`. The same tiering exists at OpenAI and other providers.

**Engineering Insight: model names are perishable.** OpenAI is shutting down `gpt-5-mini` on 11 December 2026 and names `gpt-5.6-terra` as its replacement. Providers retire models on a published schedule, which is exactly why the model ID belongs in configuration. When a model is retired, you change one line in `.env`, not every file that calls the API.

### Set up your provider

1. **Pick one provider**, OpenAI unless you have a reason to choose another, and create an API key:
   - OpenAI: [platform.openai.com/api-keys](https://platform.openai.com/api-keys)
   - Anthropic: [Claude Console](https://platform.claude.com/)
   - OpenRouter: [openrouter.ai/settings/keys](https://openrouter.ai/settings/keys)
   - Ollama: no key. [Install Ollama](https://ollama.com/download), then run `ollama pull llama3.2` (about 2 GB).
2. **Set a spending limit** or budget alert in the provider's dashboard before you write any code that calls it.
3. In the **course repository root**, copy `.env.example` to `.env`. Fill in your key, and set `LLM_PROVIDER` and `LLM_MODEL` (the file lists an example model ID for each provider). `.env` is already in `.gitignore`.
4. **Restart the kernel**, then run the cells below.

A key is a password that bills your account. Never paste it into a notebook cell, a source file or a chat window, and never print it. If one leaks, for example by being committed, revoke it in the dashboard immediately: deleting the commit is not enough.

### Reading the configuration safely

`python-dotenv`'s `load_dotenv()` reads `.env` into environment variables. (Session 05 covers configuration properly; here we only need the one call.) It does **not** overwrite variables that are already set, so a real environment variable always wins over the file.

**Predict:** if you haven't created `.env` yet, what will this cell report for the selected provider?

In [ ]:
import os

from dotenv import load_dotenv

env_file = repo_root / ".env"
load_dotenv(env_file)

PROVIDERS = {
    "openai": {"key_env": "OPENAI_API_KEY", "base_url": None, "example_model": "gpt-5.6-terra"},
    "anthropic": {"key_env": "ANTHROPIC_API_KEY", "base_url": None, "example_model": "claude-haiku-5-5"},
    "openrouter": {
        "key_env": "OPENROUTER_API_KEY",
        "base_url": "https://openrouter.ai/api/v1",
        "example_model": "openai/gpt-5.6-terra",
    },
    "ollama": {
        "key_env": None,
        "base_url": os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434/v1"),
        "example_model": "llama3.2",
    },
}


def key_status(env_name: str | None) -> str:
    if env_name is None:
        return "not needed"
    return "set" if os.environ.get(env_name) else "NOT SET"  # never print the value itself


provider = os.environ.get("LLM_PROVIDER", "openai")
if provider not in PROVIDERS:
    print(f"Unknown LLM_PROVIDER {provider!r}; expected one of {sorted(PROVIDERS)}")
else:
    model = os.environ.get("LLM_MODEL", PROVIDERS[provider]["example_model"])
    print(f".env file found: {env_file.exists()}")
    print(f"Selected:        {provider} / {model}")
    print()
    for name, settings in PROVIDERS.items():
        print(f"{name:<11} key {key_status(settings['key_env']):<11} example model {settings['example_model']}")

**Explain:** with no `.env` and no environment variables, you get the defaults: `openai / gpt-5.6-terra`, with the OpenAI key `NOT SET`. Notice that the cell reports *whether* a key is set, never what it is. Even "harmless" partial keys end up in screenshots and logs.

### One client factory, four providers

This builds a client for the selected provider **without sending anything**: creating a client makes no network call. Look at how little differs between the three OpenAI-compatible providers.

In [ ]:
import anthropic
from openai import OpenAI, OpenAIError


def make_client(provider_name: str):
    settings = PROVIDERS[provider_name]
    if provider_name == "anthropic":
        return anthropic.Anthropic()  # reads ANTHROPIC_API_KEY itself
    if settings["key_env"] is None:
        api_key = "ollama"  # Ollama ignores the key, but the SDK requires a value
    else:
        api_key = os.environ.get(settings["key_env"])
    return OpenAI(api_key=api_key, base_url=settings["base_url"])


try:
    client = make_client(provider)
    print("Client ready:", type(client).__module__.split(".")[0], "->", client.base_url)
except OpenAIError as error:
    print(f"Could not create the client: {str(error).split('.')[0]}.")
    print(f"Set {PROVIDERS[provider]['key_env']} in .env, restart the kernel, and re-run.")

**Explain:** without a key, `OpenAI()` refuses to start and raises `OpenAIError: Missing credentials` straight away. That's a *fail-fast* design: you find out at start-up, not halfway through a request. `anthropic.Anthropic()` resolves credentials lazily, so a missing key only surfaces on the first request. When you write your own clients (Session 05), decide deliberately which behaviour you want.

### Sending one request (opt-in)

The cell below sends **one short request**, but only if you change `RUN_LIVE_CALL` to `True`. Hosted providers bill per token. A one-line reply costs a fraction of a cent, but the notebook never spends your money by default. With Ollama the call is free and runs locally (start the Ollama app first).

Notice the two request shapes. The three OpenAI-compatible providers use `chat.completions.create`, and the answer is in `choices[0].message.content`. Anthropic uses `messages.create`, requires `max_tokens`, and returns a list of content blocks. (OpenAI also offers a newer *Responses* API, which Session 09 introduces.)

In [ ]:
RUN_LIVE_CALL = False  # set to True to send one request to your selected provider

PROMPT = "In one sentence, what is a Python virtual environment?"


def ask(provider_name: str, model_id: str, prompt: str) -> str:
    llm = make_client(provider_name)
    if provider_name == "anthropic":
        response = llm.messages.create(
            model=model_id, max_tokens=16000, messages=[{"role": "user", "content": prompt}]
        )
        return "".join(block.text for block in response.content if block.type == "text")
    response = llm.chat.completions.create(
        model=model_id, messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


required_key = PROVIDERS[provider]["key_env"]

if not RUN_LIVE_CALL:
    print("Skipped: set RUN_LIVE_CALL = True to send one request to", provider, "/", model)
elif required_key and not os.environ.get(required_key):
    print(f"Not sent: {required_key} is not set. Add it to .env and restart the kernel.")
else:
    print(f"{provider} / {model}:")
    print(ask(provider, model, PROMPT))

**Modify:** once your provider works, switch `LLM_PROVIDER` and `LLM_MODEL` in `.env` to a second provider (Ollama is free), restart the kernel and run Part 8 again. You change only configuration; the code stays the same.

**Common Mistake:** using one provider's model ID with another provider. Sending `claude-haiku-5-5` to OpenRouter, or `anthropic/claude-haiku-5.5` to Anthropic, fails with a "model not found" error. The fix is configuration, not code.

**Common Mistake:** creating `.env` somewhere other than the repository root, or editing it without restarting the kernel. Because `load_dotenv()` never overwrites a variable that is already set, re-running the cell after an edit keeps the *old* values. Restart the kernel instead.

## Part 9: Embeddings: OpenAI by default, fastembed locally

From Session 11 onwards you will build search and RAG features, and those rest on **embeddings**. Like Part 8, this is a setup preview (self-study, about 25 minutes), and it gives you code you can reuse in later sessions.

### The problem embeddings solve

An on-call engineer at Orbital Logistics types *"the website is down, what should I check?"* The relevant runbook says *"When the shipment tracking site returns errors, check the load balancer health first."* The two sentences share almost no words, so a keyword search (`LIKE '%website%'`, or `grep`) misses it.

An **embedding model** turns a piece of text into a list of numbers, a **vector**, such that texts with similar *meaning* get vectors that point in similar directions. You embed every document once and store the vectors. At query time you embed the question and compare it with the stored vectors. The closest ones are the most relevant documents, whatever words they use.

![From text to similarity with embeddings](../../../assets/diagrams/embedding-options.svg)

**How this works**

- Each text, document or query, goes through **one** embedding model and comes back as a fixed-length vector: 1,536 numbers for OpenAI's `text-embedding-3-small`, 384 for fastembed's `BAAI/bge-small-en-v1.5`.
- **OpenAI** (the course default) runs the model on its servers. It needs `OPENAI_API_KEY` and costs about USD 0.02 per million tokens, so embedding the whole course corpus costs cents.
- **fastembed** runs a small open model on your own CPU. It is free, needs no key and works offline after a one-time 67 MB download. Vector quality is good for its size, but larger hosted models usually retrieve better.
- **Cosine similarity** measures how closely two vectors point the same way. Ranking documents by their similarity to the query vector is the core of semantic search.
- **Failure case:** vectors from different models aren't comparable. They live in different "spaces" and often have different lengths. If you switch models, you must re-embed every document.

### Choosing an embedding provider

| | OpenAI `text-embedding-3-small` (default) | fastembed `BAAI/bge-small-en-v1.5` |
|---|---|---|
| Runs | On OpenAI's servers, via the API | On your machine (CPU, ONNX Runtime) |
| Cost | About USD 0.02 per 1M tokens | Free |
| Needs | `OPENAI_API_KEY` and network | A 67 MB model download once, then nothing |
| Vector length | 1,536 (can be shortened with `dimensions=`) | 384 |
| Data privacy | Text is sent to OpenAI | Text never leaves your machine |
| Good for | Course default; production systems already using OpenAI | Learning, offline work, tests, private documents, zero budget |

Configuration follows the same pattern as Part 8. In `.env`, `EMBEDDING_PROVIDER` is `openai` or `fastembed`, and `EMBEDDING_MODEL` holds the model ID.

> Ollama can also serve embedding models, for example `ollama pull nomic-embed-text`, through the same OpenAI-compatible `base_url` you saw in Part 8. We use fastembed as the local option here because it needs no separate server.

First, some shared setup: the configuration, a few runbook snippets and a query, and the cosine-similarity function both providers will use.

In [ ]:
import numpy as np

EMBEDDING_DEFAULTS = {"openai": "text-embedding-3-small", "fastembed": "BAAI/bge-small-en-v1.5"}

embedding_provider = os.environ.get("EMBEDDING_PROVIDER", "openai")
embedding_model = os.environ.get("EMBEDDING_MODEL") or EMBEDDING_DEFAULTS.get(embedding_provider)
print(f"Embedding provider: {embedding_provider} / {embedding_model}")

RUNBOOK_SNIPPETS = [
    "If the order database stops responding, fail over to the replica and page the on-call DBA.",
    "Warehouse scanners must be recharged at the end of every shift.",
    "Rotate API credentials every 90 days and never store them in source code.",
    "When the shipment tracking site returns errors, check the load balancer health first.",
    "Annual leave requests need manager approval two weeks in advance.",
]
QUERY = "the website is down, what should I check?"


def cosine_similarity(matrix: np.ndarray, vector: np.ndarray) -> np.ndarray:
    """Similarity of each row of `matrix` to `vector`, from -1 to 1."""
    return matrix @ vector / (np.linalg.norm(matrix, axis=1) * np.linalg.norm(vector))


def show_ranking(document_vectors: np.ndarray, query_vector: np.ndarray, texts: list[str]) -> None:
    scores = cosine_similarity(document_vectors, query_vector)
    for score, text in sorted(zip(scores, texts), reverse=True):
        print(f"{score:.3f}  {text}")

`matrix @ vector` is a matrix–vector product: one dot product per document, computed in a single NumPy operation. NumPy is the standard numerical-array library. It came into the course environment as a dependency of fastembed, and you'll use it properly in Session 11.

### OpenAI embeddings (default, opt-in call)

The call shape is short: `client.embeddings.create(model=..., input=[...])`, with one `embedding` per input in `response.data`, in input order. Send all texts in **one** request rather than one request per text. The API accepts up to 2,048 inputs per request.

In [ ]:
RUN_OPENAI_EMBEDDINGS = False  # set to True to send one small, paid request


def embed_with_openai(texts: list[str], model_id: str = "text-embedding-3-small") -> tuple[np.ndarray, int]:
    response = OpenAI().embeddings.create(model=model_id, input=texts)
    vectors = np.array([item.embedding for item in response.data])
    return vectors, response.usage.total_tokens


if not RUN_OPENAI_EMBEDDINGS:
    print("Skipped: set RUN_OPENAI_EMBEDDINGS = True to embed the snippets with OpenAI")
elif not os.environ.get("OPENAI_API_KEY"):
    print("Not sent: OPENAI_API_KEY is not set. Add it to .env and restart the kernel.")
else:
    vectors, tokens_used = embed_with_openai(RUNBOOK_SNIPPETS + [QUERY])
    print(f"Vectors: {vectors.shape}  tokens: {tokens_used}  cost: about ${tokens_used * 0.02 / 1_000_000:.6f}")
    show_ranking(vectors[:-1], vectors[-1], RUNBOOK_SNIPPETS)

Notice that the query was embedded **in the same request** as the documents (it's the last row, `vectors[-1]`). One round trip, one bill line.

### fastembed in depth

[fastembed](https://qdrant.github.io/fastembed/) is an open-source library from Qdrant, the vector-database company. It runs embedding models locally using **ONNX Runtime**, an optimised inference engine for CPUs. That means no PyTorch and no GPU: the whole install is a few tens of MB, and a small model embeds hundreds of sentences per second on a laptop.

What you need to know to use it well:

| Topic | What to know |
|---|---|
| Install | It's already in the course environment. In your own projects, run `uv add fastembed`. |
| Main class | `from fastembed import TextEmbedding`. The default model is `BAAI/bge-small-en-v1.5`. |
| First run | Downloads the model (67 MB for the default) from Hugging Face. That can take a few minutes on a slow connection; afterwards it loads in about a second. |
| Model cache | **By default the cache is in your system's temp folder**, which the OS may clean up. Pass `cache_dir=` (as below) or set `FASTEMBED_CACHE_PATH`. |
| Output | `embed()` returns a **generator** of NumPy arrays. Nothing is computed until you consume it, for example with `list(...)` or `np.array(list(...))`. |
| Queries vs documents | Use `query_embed()` for search queries and `passage_embed()` for documents. For `bge-small-en-v1.5` all three methods give identical vectors (we checked), but some models treat queries and documents differently, and these methods handle that for you. |
| Batching | `embed(texts, batch_size=256)` by default. Larger corpora also accept `parallel=` to use several CPU cores. |
| Windows warning | You may see a `huggingface_hub` warning about symlinks. It is harmless; the cache just uses a little more disk. Set `HF_HUB_DISABLE_SYMLINKS_WARNING=1` to hide it. |

**Choosing a model.** fastembed supports dozens of models. Listing them needs no download:

In [ ]:
from fastembed import TextEmbedding

catalogue = sorted(TextEmbedding.list_supported_models(), key=lambda entry: entry["size_in_GB"])
print(f"{len(catalogue)} text embedding models. The smallest ones:\n")
for entry in catalogue[:8]:
    print(f"{entry['model']:<45} {entry['dim']:>5} dims  {entry['size_in_GB'] * 1000:>5.0f} MB")

`BAAI/bge-small-en-v1.5` is a good default: small, English, and widely used for retrieval. If retrieval quality matters more than size, `BAAI/bge-base-en-v1.5` (768 dimensions, about 210 MB) is a common step up.

**Try it.** The next cell loads the model. The first run downloads about 67 MB into `~/.cache/fastembed` (or your `FASTEMBED_CACHE_PATH`), and later runs load from disk. Set `RUN_FASTEMBED = True` to try it. It's free.

In [ ]:
RUN_FASTEMBED = False  # set to True: free, local; first run downloads about 67 MB

fastembed_cache = Path(os.environ.get("FASTEMBED_CACHE_PATH") or Path.home() / ".cache" / "fastembed")

if RUN_FASTEMBED:
    local_model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5", cache_dir=str(fastembed_cache))
    print(f"Loaded {local_model.model_name}: {local_model.embedding_size} dimensions, cached in {fastembed_cache}")
else:
    print("Skipped: set RUN_FASTEMBED = True to download (once) and load the local model")

**Predict:** which runbook snippet will rank first for *"the website is down, what should I check?"*, and which last?

In [ ]:
if RUN_FASTEMBED:
    document_vectors = np.array(list(local_model.passage_embed(RUNBOOK_SNIPPETS)))
    query_vector = next(iter(local_model.query_embed(QUERY)))  # a generator, even for one query

    print("Document vectors:", document_vectors.shape, " query vector:", query_vector.shape)
    print("First five numbers of the query vector:", np.round(query_vector[:5], 3))
    print("Vector lengths:", np.round(np.linalg.norm(document_vectors, axis=1), 3))
    print()
    show_ranking(document_vectors, query_vector, RUNBOOK_SNIPPETS)
else:
    print("Skipped: set RUN_FASTEMBED = True in the cell above and run both cells")

**Explain:**

- The tracking-site snippet ranks first (about 0.63 when we ran it) despite sharing almost no words with the query. The model connects "website is down" with "site returns errors". The database outage comes second, because it is also an outage. Leave policy and warehouse scanners rank last.
- The individual numbers mean nothing on their own. Only the **relationships** between vectors carry meaning.
- Every vector has length 1.0: this model returns **normalised** vectors, so cosine similarity equals a plain dot product. Our `cosine_similarity` divides by the lengths anyway, so it stays correct for any model.
- Scores are relative. 0.63 isn't "63% relevant"; it's simply higher than the others for this query and this model. Don't hard-code thresholds without testing them on your own data.

**Modify:**

1. Change `QUERY` to *"how should we store passwords?"* and predict the new top result before running.
2. Add a snippet of your own to `RUNBOOK_SNIPPETS` (one you'd expect to win for some query) and test it.
3. Load `BAAI/bge-base-en-v1.5` instead (about 210 MB). Do the scores change? Does the *ranking* change?

### A reusable helper for later sessions

This is the pattern later sessions build on: one function and one configuration, two providers. `lru_cache` makes sure the local model is loaded only once per process, because loading it on every call would cost a second or more each time.

In [ ]:
from functools import lru_cache


@lru_cache(maxsize=None)
def load_local_embedding_model(model_id: str) -> TextEmbedding:
    return TextEmbedding(model_name=model_id, cache_dir=str(fastembed_cache))


def embed_texts(texts: list[str], provider: str = "openai", model_id: str | None = None) -> np.ndarray:
    """Return one row per text, using the configured embedding provider."""
    model_id = model_id or EMBEDDING_DEFAULTS[provider]
    if provider == "openai":
        response = OpenAI().embeddings.create(model=model_id, input=texts)
        return np.array([item.embedding for item in response.data])
    if provider == "fastembed":
        return np.array(list(load_local_embedding_model(model_id).embed(texts)))
    raise ValueError(f"Unknown embedding provider {provider!r}; expected 'openai' or 'fastembed'")


print("embed_texts() is ready. Example: embed_texts(RUNBOOK_SNIPPETS, provider='fastembed')")

**Engineering Insight: record which model made each vector.** When you store embeddings (Session 11 onwards), store the model ID alongside them. When someone later changes `EMBEDDING_MODEL`, your code can detect the mismatch and re-embed, instead of silently returning nonsense or crashing. If you do mix them, the crash looks like this, because 384 numbers can't be multiplied with 1,536:

```text
ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 1536 is different from 384)
```

**Common Mistake:** forgetting that `embed()` returns a generator. `vectors = local_model.embed(texts)` computes nothing. Wrap it in `list(...)`. A generator can also be consumed only **once**, so a second loop over it finds it empty.

**Common Mistake:** leaving fastembed's cache in the system temp folder. The model silently re-downloads after the OS cleans up, often on the day you're offline.

## Common mistakes

| Mistake | Why it hurts | Instead |
|---|---|---|
| Installing packages globally with `pip install` | Projects silently share, and break, each other's dependencies | One project, one `.venv`, managed with `uv add` |
| Running `python main.py` from an unconfigured terminal | You may get a different interpreter and different packages | `uv run main.py` |
| Committing `.venv/` | Huge, machine-specific, and breaks on other operating systems | Commit `uv.lock`; anyone can rebuild `.venv` with `uv sync` |
| Editing `uv.lock` by hand | It will be overwritten, or become inconsistent with `pyproject.toml` | Change `pyproject.toml` through `uv add`/`uv remove`, then let uv relock |
| Not committing `uv.lock` | Every machine resolves "latest", and builds drift | Always commit it |
| Trusting type hints to reject bad data | Hints are not enforced at runtime | Validate external data (Session 06) |
| Assuming a script works because it ran once | Run-time errors hide in branches that didn't run | Test each path (Session 04) |
| Hard-coding an API key or model ID | Keys leak through git; model IDs get retired | Read both from the environment (`.env`), as in Part 8 |
| Comparing embeddings made by different models | Different vector spaces (and lengths): wrong results or a crash | Store the model ID with the vectors; re-embed when the model changes |

## Key takeaways

- CPython compiles your code to bytecode automatically, then interprets it. Only **syntax** errors are caught up front; everything else appears when the line runs.
- Imports search `sys.path`, and **which interpreter is running decides which packages are visible.**
- A virtual environment is a per-project `site-packages` plus a pointer to a shared interpreter. It is cheap, and disposable.
- `pyproject.toml` is what you want, `uv.lock` is exactly what you get, and `.venv/` is a rebuildable copy of the lock. Commit the first two; never commit the third.
- `uv run` removes a whole class of "wrong environment" bugs. Make it a habit.
- LLM provider, model and key are **configuration**. The `openai` SDK reaches OpenAI, OpenRouter and Ollama by changing `base_url`; Anthropic uses its own SDK. The course default is OpenAI `gpt-5.6-terra`.
- Embeddings turn text into vectors, so you can search by meaning. OpenAI `text-embedding-3-small` is the default; fastembed's `BAAI/bge-small-en-v1.5` runs locally for free. Never mix vectors from different models.

## Further reading

The curated, official references for this session are in [`../resources/references.md`](../resources/references.md). Next, test yourself with **`02-practice-quiz.ipynb`**, then start the workbook assignment.